# 03 · Tokens, context windows and sampling dials
Level L1 · Part 1 · core · ~20 min · builds on 02

## Goal
Count the tokens of a call, price it, see how much of the context window it uses, and see what the
sampling and thinking dials change, and which models lock them.

## Why it matters
Without a way to count tokens, the Brightside support desk from lesson 01 guesses the bill and the room
left in a call. A guess from the number of characters is only a rule of thumb, and a budget built on it is
wrong by an amount you cannot see from the text. Here is the guess, using the common rule of thumb of about
four characters per token. The policy is a short text written for this lesson (not a real Brightside
document). Step 1 asks the model to count the policy's tokens and prints that count next to the guess.

In [1]:
POLICY = """Brightside Outfitters returns policy (made up for this lesson).
Items can be returned within 30 days of delivery. The item must be unused, in its original packaging,
and with the tags on. Start a return from your order page; we send a prepaid label by email within one
working day. Refunds go back to the original payment method within 5 to 10 working days after the
returned parcel is scanned at our warehouse. Delivery charges are not refunded unless the item arrived
damaged or the wrong item was sent. Sale items marked final sale cannot be returned. Gift cards are not
refundable; we issue store credit instead. For a lost parcel, contact the desk after 10 working days
from dispatch; we will trace the parcel or send a replacement or refund once the carrier confirms the loss."""
print("characters:", len(POLICY), "· guess at 4 characters per token:", len(POLICY) // 4, "tokens")

characters: 782 · guess at 4 characters per token: 195 tokens


The guess is a starting point, not a bill. To know what the model reads, ask the model's own tokenizer.
Each vendor has its own tokenizer, so a count from another vendor's library is a different number.
First, the settings that every cell below uses. They are plain numbers, so no key is needed here.

In [2]:
MODEL = "gemini-3.1-flash-lite"       # the course's pinned model; the .env override applies in the setup cell
PRICE_IN, PRICE_OUT = 0.25, 1.50       # USD per 1M tokens, Gemini pricing page, checked 2026-10-20 (lesson 02)
CONTEXT_LIMIT = 1_048_576              # input tokens this model accepts, model page, checked 2026-10-20
print(f"model {MODEL} · ${PRICE_IN} in and ${PRICE_OUT} out per 1M tokens · input limit {CONTEXT_LIMIT:,} tokens")

model gemini-3.1-flash-lite · $0.25 in and $1.5 out per 1M tokens · input limit 1,048,576 tokens


## The idea
A **token** is a chunk of text, often a word or part of one. The **context window** is the room a call has:
the **input** must fit its limit, and the **output** has its own cap (`max_output_tokens`) that thinking
tokens count against. A **sampling dial** decides how the model picks the next token. The model gives each
possible next token a raw score (a **logit**). **Softmax** turns the scores into probabilities that add up to 1.
**Temperature** divides the scores before softmax: low sharpens them, high flattens them. **Top-p** keeps only
the most likely tokens that add up to p. **Thinking effort** is a third dial: more thinking, more billed tokens.

Analogy: the context window is the support agent's desk. The policy and the question must fit on it at once,
and the answer has its own sheet. Temperature is how boldly the agent picks: low means the top word, high a long shot.

## Picture
```mermaid
flowchart LR
  T[Text] --> K[Tokens: chunks with ids]
  K --> W[Input: must fit the input limit]
  W --> M[Model: a score for each possible next token]
  M --> S[Sampler: temperature and top-p pick one]
  S --> O[One token out, then repeat, within the output cap]
```

## Step by step
**Step 1 · count the tokens.** Set up the client. The key is read from `.env` (never typed here); when the
recording replays, no key is needed. Then ask the model to count the policy's tokens. Counting is a request
too, so it is in the recording. The first run with a key records each model call into
`data/llm_cassettes/`, and later runs replay that recording with no key. To record again, delete that file
and run with `TUT_LLM_MODE=record`.

In [3]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("03_tokens-context-sampling")

from google import genai
from google.genai import errors, types
from tenacity import retry, retry_if_exception, stop_after_attempt, wait_exponential_jitter
from tut_env import env

MODEL = env("TUT_LLM_MODEL", MODEL)   # the .env override (if set) replaces the pinned id
NO_KEY = "replay"                     # while replaying, the recording answers; this placeholder is never sent
client = genai.Client(
    vertexai=True,
    api_key=NO_KEY if llm_replay.is_replaying() else env("GOOGLE_CLOUD_API_KEY"),
    http_options=types.HttpOptions(timeout=30_000, retry_options=types.HttpRetryOptions(attempts=1)),
)
TRANSIENT = {408, 429, 500, 502, 503, 504}


@retry(retry=retry_if_exception(lambda e: isinstance(e, errors.APIError) and e.code in TRANSIENT),
       stop=stop_after_attempt(4), wait=wait_exponential_jitter(initial=5, max=30, jitter=5), reraise=True)
def ask(**kwargs):
    """One model call, retried on transient errors (the same wrapper as lesson 02).
    stop_after_attempt(4) means 4 attempts in total: the first try and 3 retries."""
    return client.models.generate_content(**kwargs)


counted = client.models.count_tokens(model=MODEL, contents=POLICY)
policy_tokens = counted.total_tokens
print(f"Gemini counts {policy_tokens} tokens for {len(POLICY)} characters "
      f"(the guess was {len(POLICY) // 4}).")

LLM mode: replay (03_tokens-context-sampling.yaml)


Gemini counts 168 tokens for 782 characters (the guess was 195).


**Step 2 · the context window.** The limit applies to the input: this model accepts up to 1,048,576 input
tokens. The output has its own cap, set with `max_output_tokens`, and thinking tokens count against that cap.
The next cell shows how much of the input limit the policy takes. If Step 1 was skipped, the cell falls back
to the guess, so the rest of the lesson still runs.

In [4]:
policy_tokens = globals().get("policy_tokens", len(POLICY) // 4)
print(f"input limit: {CONTEXT_LIMIT:,} tokens")
print(f"the policy uses {policy_tokens / CONTEXT_LIMIT:.4%} of it")
print(f"copies of the policy that would fit in one call: {CONTEXT_LIMIT // policy_tokens:,}")
print("The output has its own cap (max_output_tokens); thinking tokens count against that cap.")

input limit: 1,048,576 tokens
the policy uses 0.0160% of it
copies of the policy that would fit in one call: 6,241
The output has its own cap (max_output_tokens); thinking tokens count against that cap.


**Step 3 · the thinking dial.** Some models think before they answer. Gemini 3 models take a `thinking_level`;
other models differ (Claude calls the same dial `effort`), so check each SDK. Thinking tokens count against
`max_output_tokens`, are billed as output, and are not shown in the answer. The usage record has three counts:
`prompt_token_count` is the input you sent, `candidates_token_count` is the visible answer, and
`thoughts_token_count` is the thinking the answer hides. `temperature=0` is set so that both runs use the same
repeatable settings: the thinking level is the only thing that changes, so the bill is what you compare.
Both answers should be the same number, $30.

In [5]:
QUESTION = ("Order 1093 cost $84 including $6 delivery. The customer returns the $30 item and keeps the rest. "
            "Delivery is not refunded. How much is refunded? Answer with the number and one line of working.")


def run_at(level):
    cfg = types.GenerateContentConfig(temperature=0, max_output_tokens=1024,
                                      thinking_config=types.ThinkingConfig(thinking_level=level))
    return ask(model=MODEL, contents=QUESTION, config=cfg)


def show(name, resp):
    u = resp.usage_metadata
    thinking = u.thoughts_token_count or 0
    out = (u.candidates_token_count or 0) + thinking
    usd = (u.prompt_token_count * PRICE_IN + out * PRICE_OUT) / 1_000_000
    print(f"{name:<6} answer: {resp.text.strip().splitlines()[0]}")
    print(f"       input {u.prompt_token_count} · thinking {thinking} · output {u.candidates_token_count} · ${usd:.6f}")


show("low", run_at(types.ThinkingLevel.LOW))
show("high", run_at(types.ThinkingLevel.HIGH))

low    answer: $30
       input 48 · thinking 121 · output 27 · $0.000234
high   answer: $30
       input 48 · thinking 459 · output 28 · $0.000742


**Step 4 · the sampling dials, on a toy sampler.** No model is needed to see how temperature works. The
scores below are made up for the next word after "Your refund will arrive in ...". They are logits (raw
scores). Softmax turns them into probabilities that add up to 1. Temperature divides each score before softmax:
at 0.2 the top word becomes almost certain, at 2.0 the probability spreads out.

Temperature 0 means greedy: always take the most likely token. The toy function cannot take 0, because it
divides the scores by the temperature and dividing by zero is undefined. So the sharpest row here is 0.2,
which is already nearly greedy.

In [6]:
import math

logits = {"days": 2.0, "weeks": 1.0, "hours": 0.2, "never": -1.5}   # made-up scores for one next word


def softmax_t(scores, temperature):
    top = max(v / temperature for v in scores.values())             # subtract the max for stable maths
    e = {k: math.exp(v / temperature - top) for k, v in scores.items()}
    total = sum(e.values())
    return {k: x / total for k, x in e.items()}


for t in (0.2, 1.0, 2.0):
    probs = softmax_t(logits, t)
    print(f"temperature {t}: " + ", ".join(f"{k} {p:.0%}" for k, p in probs.items()))

temperature 0.2: days 99%, weeks 1%, hours 0%, never 0%
temperature 1.0: days 64%, weeks 24%, hours 11%, never 2%
temperature 2.0: days 46%, weeks 28%, hours 19%, never 8%


**Top-p** keeps the most likely words until their probabilities add up to p, and drops the rest. The model
then picks only from that kept set.

In [7]:
def top_p_set(probs, p):
    kept, total = [], 0.0
    for word, pr in sorted(probs.items(), key=lambda kv: -kv[1]):
        kept.append(word)
        total += pr
        if total >= p:
            break
    return kept


print("top-p 0.9 at temperature 1.0 keeps:", top_p_set(softmax_t(logits, 1.0), 0.9))
print("top-p 0.5 at temperature 1.0 keeps:", top_p_set(softmax_t(logits, 1.0), 0.5))

top-p 0.9 at temperature 1.0 keeps: ['days', 'weeks', 'hours']
top-p 0.5 at temperature 1.0 keeps: ['days']


**Step 5 · the same dials on a local model.** Ollama runs a small open model on your machine, for free. You
need Ollama installed and running, the Python client (`pip install ollama`, already in `requirements.txt`), and
the model pulled once with `ollama pull gemma3:1b`. The model has about 1 billion parameters and is under 1 GB on
disk. This cell can be skipped (it is tagged `needs-download`); the saved notebook then shows the recorded output.
Run one prompt at temperature 0, then at a higher temperature with top-p. The seed (7) makes the run repeatable
only on the same Ollama and model version. In the recorded run the higher temperature changed the wording, not
the fact: both sentences say a parcel can arrive later than expected. Ollama reports the tokens it read
(`prompt_eval_count`) and wrote (`eval_count`).

In [8]:
import ollama

local = ollama.Client(host=env("OLLAMA_BASE_URL", "http://localhost:11434"))
LOCAL_MODEL = "gemma3:1b"
prompt = "Tell a customer in one sentence why a parcel can be late."
for temperature, top_p in ((0.0, 1.0), (1.5, 0.9)):
    r = local.chat(model=LOCAL_MODEL, messages=[{"role": "user", "content": prompt}],
                   options={"temperature": temperature, "top_p": top_p, "seed": 7, "num_predict": 80})
    print(f"temperature {temperature}, top_p {top_p}: {r.message.content.strip()}")
    print(f"  tokens in {r.prompt_eval_count} · out {r.eval_count} · stop: {r.done_reason}")

llm_replay.stop()

⏭  Skipped: this cell needs [download]. Run it yourself, or re-run with TUT_ENABLE=download.


**Step 6 · which models lock the dials.** Check the model's page before you tune a dial. Claude Haiku 5.5
(2026-10-07) accepts only `temperature=1`, Claude 4.7 and later omit the sampling dials, and GPT-6 Astra has no
temperature, top-p or logprobs. The Gemini page for this course's model does not state temperature or top-p
support, so check it before you rely on them. Ollama honours all of them. Keep the settings in one place, so a
model change does not silently change your behaviour. Source: RESEARCH.md, "Models for lessons". Fine-tuning and
distillation change a model, not a setting; this lesson does not build either (Go deeper).

## At work
- The biggest line of a support bill is often the same policy pasted into every call. Teams count tokens
  per request and cap input and output per day → lesson 33 (cost and latency, later in the course).
- Long prompts make models miss facts (context rot). Teams trim, summarise or fetch only the passages a
  question needs → lesson 15 (context engineering, later in the course) and lesson 17 (RAG, later in the course).
- Thinking is billed as output. Teams choose the lowest thinking level that passes their evals, and check
  the change with a regression gate → lesson 28 (evals in CI, later in the course).
- Low temperature is the usual setting for extraction and tool arguments, because the output must match a
  schema → lesson 04 (structured output).

## Common mistakes
1. **Counting characters as tokens.** The rule of thumb is a guess. Count with the model's tokenizer (Step 1).
2. **Forgetting that thinking uses the output cap.** Thinking and the answer share `max_output_tokens`, so a
   cap that is too small can cut the answer short.
3. **Paying only for the visible answer.** The Step 3 bill includes the thinking tokens the answer hides.
4. **A wrong setting type.** A temperature written as a word is caught on your machine. Pass a number.

The SDK catches a wrong type locally. The `ValidationError` below comes from pydantic, the data-validation library
the SDK uses to check its settings before a request is sent. A locked model rejects the dial at the API instead:
Claude Haiku 5.5 errors on any temperature other than 1. This Gemini call does not lock it, so the cell shows the local check only.

In [9]:
types.GenerateContentConfig(temperature="hot")

ValidationError: 1 validation error for GenerateContentConfig
temperature
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='hot', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/float_parsing

## Try it
The desk expects 2,000 chats a day, each sending the policy (`policy_tokens` from Step 1, which is 168 tokens
here) plus 60 tokens of customer message, and each getting 150 output tokens back. Work out the monthly cost
for 30 days with `PRICE_IN` and `PRICE_OUT`. Then add the cost of 400 extra thinking tokens per chat, billed at
the output price. The cost of a call is `(input tokens × PRICE_IN + output tokens × PRICE_OUT) / 1,000,000`.
The printed monthly cost should be $16.92. If Step 1 was skipped, the fallback uses the character guess, which
is 195 tokens, and the monthly cost is $17.32.

In [10]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [11]:
policy_tokens = globals().get("policy_tokens", len(POLICY) // 4)
chats = 2000 * 30
month_in = chats * (policy_tokens + 60)
month_out = chats * 150
print(f"per month: ${(month_in * PRICE_IN + month_out * PRICE_OUT) / 1_000_000:,.2f}")
print(f"extra for 400 thinking tokens per chat: ${chats * 400 * PRICE_OUT / 1_000_000:,.2f}")

per month: $16.92
extra for 400 thinking tokens per chat: $36.00


With the policy at 168 tokens, the month costs $16.92 and the thinking adds $36.00. With the fallback of 195
tokens, the month costs $17.32. The policy is most of the input, so trimming it is the cheapest saving.

## Recap
- A token is a chunk of text. Count it with the model's tokenizer, not with characters.
- The input must fit the input limit. The output has its own cap, and thinking tokens count against it.
- Temperature and top-p change which token is picked. Temperature 0 means always the most likely token; higher temperature is more varied.
- Thinking is a dial billed as output, and some models lock the sampling dials.

**Interview:** "A teammate says to fix wrong answers by setting temperature to 0. What do you check first?"
Strong answer: "Which model and settings are really sent and whether the model accepts them, and whether the
wrong answers come from missing context or a too-small output cap. Temperature 0 makes output steadier, not more
correct, so I'd fix the context and check the answers with evals."